<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/08%20-%20Sequence%20Models%20and%20Transformers/08a_smiles_language_model.ipynb)

# 08a · 让模型写分子：SMILES 序列语言模型与基线对比

前面几讲讨论的模型大多属于**判别式模型（Discriminative Models）**：给定一个已知的分子结构或材料显微图像，预测它的溶解度、相分类或能量上限。从本讲开始，我们把问题反过来：**如果不给具体分子输入，能不能让模型自己“写出”符合化学语法的新分子，或者根据反应物序列自动写出产物序列？**

由于分子可以表示为 SMILES 字符串，最直接的思路就是借鉴自然语言处理中的**自回归语言模型（Autoregressive Language Model）**：把 SMILES 看作一句话，把原子、化学键和成环数字看作词元（token），训练神经网络根据前文逐个预测下一个词元。

本节将以**哈佛清洁能源计划（CEP）有机光伏候选分子数据集**为例，从分词器与非神经网络基线（二元语法模型）出发，使用 PyTorch 从头搭建并训练一个基于长短期记忆网络（LSTM）的 SMILES 语言模型，系统检验生成分子的有效性、唯一性、新颖性以及化学结构分布。

**本节核心内容：**
1. 自回归分子生成的基本思想与 CEP 有机光伏数据集
2. SMILES 正则分词（Tokenization）与词表编码
3. 非神经网络基线：二元模型（Bigram）为什么学不会长程成环约束
4. 构建 SMILES 语言模型（`Embedding` + `LSTM` + `Linear`）与 Teacher Forcing 训练目标
5. 模型训练、梯度裁剪与预训练权重加载
6. 自回归随机采样、温度系数调节与无效分子的错误诊断
7. 模型真的学到化学了吗？性质分布、共轭构筑单元频率与最近邻相似度检验

> **运行时间提示**：第 5 节会在 CPU 上现场训练 3 轮（约 2–3 分钟）以展示有效率随训练轮数的攀升过程；后续深入评估将直接加载预先训练满 20 轮的模型权重（`data/cep_rnn.pt`）。课内练习的完整参考答案见同目录下的 `solutions/08_sequence_solutions.ipynb`。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "08 - Sequence Models and Transformers"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import re
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors

RDLogger.DisableLog("rdApp.*")
torch.manual_seed(0)
np.random.seed(0)
print("PyTorch version:", torch.__version__)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---
## 1. 自回归分子生成在做什么？

自回归生成模型的核心思想与输入法联想词完全一致：

> 阅读大量合法句子之后，给定已经写出的前几个词元，估计下一个词元出现的条件概率分布。

把“句子”换成 SMILES 字符串，把“字”换成原子与化学键符号，就得到了一个**分子语言模型**。从起始符开始，反复执行“预测下一个词元的概率分布 $\rightarrow$ 按概率抽样 $\rightarrow$ 将抽出的词元追加到末尾”，就能逐个字符地写出一整个新分子。

在动手写代码之前，需要牢记生成模型最核心的一条规律：

> **训练一个无条件分子生成模型，本质上不是教它“理解物理化学原理”，而是教它拟合训练数据的概率分布。**

训练集里是什么类型的分子，模型采样出的分子就会呈现出什么统计特征。训练集是水溶性小分子药物，它就生成类药小分子；训练集是有机光伏共轭分子，它就生成稠环共轭分子。模型本身不会凭空产生超出训练分布的化学直觉——第 7 节我们将通过定量统计验证这一点。

### 1.1 数据集：哈佛清洁能源计划（CEP）有机光伏分子

本节使用的数据集来自 **CEP（Harvard Clean Energy Project）** 子集：包含约 3 万个有机光伏（OPV）给体候选分子，每个分子标注了基于 DFT 计算的 HOMO/LUMO 能级并通过 Scharber 模型估算出的**能量转换效率上限（PCE, %）**。

选择这批数据作为序列生成入门案例有两个原因：

1. **鲜明的材料化学背景**：这批分子由噻吩、呋喃、硒吩、苯并噻二唑、咔唑、硅杂环戊二烯等典型有机半导体共轭单元拼接而成；
2. **词表紧凑、结构规律清晰**：它们主要由稠合或偶联的共轭环系组成，适合在课堂时间内用 CPU 训练出高有效率的序列生成模型。

*注：Scharber 模型的 PCE 是仅依赖分子轨道能级的理论上限估计，真实光伏器件效率还高度依赖薄膜给受体相分离形貌、结晶度与界面电荷提取工艺。在后续生成优化任务中，我们将其视为一个与分子共轭电子结构强相关的目标性质。*

In [ ]:
DATA_DIR = "data"
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    DATA_DIR = "."
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    try:
        from google.colab import files
        print("请上传 cep.csv（如需使用预训练模型，请同时上传 cep_rnn.pt）")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 cep.csv，请确保它位于 data/ 目录下")

df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
print(f"成功读取 CEP 数据集，共 {len(df)} 个分子")
df.head()

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
heavy_counts = [m.GetNumHeavyAtoms() for m in mols]
print(f"可被 RDKit 正常解析的分子数: {sum(m is not None for m in mols)} / {len(df)}")
print(f"重原子数: 平均 {np.mean(heavy_counts):.1f}，范围 {min(heavy_counts)} - {max(heavy_counts)}")
print(f"SMILES 字符长度: 平均 {df['smiles'].str.len().mean():.1f}，最大 {df['smiles'].str.len().max()}")

Draw.MolsToGridImage(mols[:8], molsPerRow=4, subImgSize=(260, 200),
                     legends=[f"PCE {p:.1f}%" for p in df["PCE"][:8]])

打印前 5 个分子的 SMILES 字符串，观察它们与第 01–03 讲使用的 AqSolDB 水溶解度分子相比有什么结构特征：

In [ ]:
for s in df["smiles"][:5]:
    print(s)

可以看到，这些 SMILES 几乎全部由小写字母（芳香原子 `c`、`n`、`o`、`s`）、数字（成环配对标记 `1`–`5`）、圆括号（支链）以及方括号杂原子（`[nH]`、`[SiH2]`、`[se]`）构成，极少出现复杂的脂肪族支链或含氧酸碱官能团。

---
## 2. SMILES 正则分词与词表构建

神经网络不能直接处理原始字符串，必须先将 SMILES 切分为一个个最小语义单元——**词元（token）**。如果直接按单个 ASCII 字符切分，会破坏两类化学符号：

1. 双字母元素符号如 `Cl`（氯）、`Br`（溴）会被拆成 `C` + `l` 和 `B` + `r`，导致模型将溴误认为硼原子（`B`）；
2. 方括号修饰的整体原子环境如 `[nH]`、`[SiH2]`、`[se]` 以及两位数的环编号（如 `%10`）一旦拆开就失去了原有的语法定义。

在化学信息学中，标准做法是使用**正则表达式（Regular Expression）**优先匹配方括号整体、`Br`/`Cl` 和 `%dd`，其余位置再按单字符切分。切分后务必加上一句断言 `assert "".join(tokens) == smiles`，确保分词器没有漏掉任何未知字符。

In [ ]:
TOKEN_PATTERN = re.compile(r"(\[[^\]]+\]|Br|Cl|[BCNOPSFIbcnops]|\(|\)|\.|=|#|-|\+|\\|/|:|%\d{2}|\d)")

def tokenize(smiles):
    tokens = TOKEN_PATTERN.findall(smiles)
    assert "".join(tokens) == smiles, f"分词结果无法拼回原字符串: {smiles}"
    return tokens

print("噻吩:    ", tokenize("c1ccsc1"))
print("OPV 分子:", tokenize("[nH]1c(cc2c3nsnc3c3c4ccccc4oc3c12)-c1ccco1"))
print("阿司匹林:", tokenize("CC(=O)Oc1ccccc1C(=O)O"))

统计整个 CEP 数据集中出现过的所有词元及其频次：

In [ ]:
token_lists = [tokenize(s) for s in df["smiles"]]
counter = collections.Counter(t for ts in token_lists for t in ts)
print(f"数据集共包含 {len(counter)} 种化学词元")
pd.Series(counter).sort_values(ascending=False).to_frame("count")

整个 3 万分子的数据集只用到了 20 种化学词元。为了将这些不定长的词元序列打包成二维张量送入神经网络，还需要在词表（vocabulary）中加入三个特殊控制词元：

- `<pad>`（编号 `0`）：填充符，用于将同一批次内较短的序列补齐到相同长度；
- `^`（编号 `1`）：序列起始符（BOS），采样时作为喂给模型的第一步输入；
- `$`（编号 `2`）：序列结束符（EOS），当模型采样输出 `$` 时，代表当前分子已经书写完毕。

In [ ]:
vocab = ["<pad>", "^", "$"] + sorted(counter)
stoi = {t: i for i, t in enumerate(vocab)}
max_len = max(len(ts) for ts in token_lists) + 2      # +2 为首尾的 ^ 和 $ 预留位置
print(f"总词表大小: {len(vocab)}，编码后最大序列长度: {max_len}")
print("词表内容:", vocab)

data = torch.zeros(len(token_lists), max_len, dtype=torch.long)
for i, tokens in enumerate(token_lists):
    ids = [stoi["^"]] + [stoi[t] for t in tokens] + [stoi["$"]]
    data[i, :len(ids)] = torch.tensor(ids)

print("\n第一个分子的原始 SMILES:", df["smiles"][0])
print("编码后的前 20 个整数编号 (0 代表 <pad>):", data[0][:20].tolist(), "...")

---
## 3. 非神经网络基线：二元语法模型（Bigram）

正如第 02 讲在引入监督学习模型前先用“永远猜训练集均值”作为基准线，构建序列生成模型时，我们也先搭建一个最简单的统计基线：**二元模型（Bigram Model）**。

二元模型的假设非常简单：预测下一个词元时，**只看当前刚写出的那一个词元** $s_{t-1}$，统计训练集中紧跟在 $s_{t-1}$ 后面的各个词元出现的经验频率 $P(s_t \mid s_{t-1})$，并按该频率随机抽签。

既然二元模型采样的每一步相邻转移 $(s_{t-1} \rightarrow s_t)$ 都在真实训练集中高频出现过，它能写出合法的 SMILES 分子吗？我们采样 1000 条序列来检验。

In [ ]:
bigram = collections.defaultdict(collections.Counter)
for tokens in token_lists:
    seq = ["^"] + tokens + ["$"]
    for a, b in zip(seq, seq[1:]):
        bigram[a][b] += 1

rng = np.random.RandomState(0)

def bigram_sample(max_steps=max_len):
    out, current = [], "^"
    for _ in range(max_steps):
        options = list(bigram[current])
        probs = np.array([bigram[current][o] for o in options], dtype=float)
        current = options[rng.choice(len(options), p=probs / probs.sum())]
        if current == "$":
            break
        out.append(current)
    return "".join(out)

print("二元模型采样的 5 条序列:")
for _ in range(5):
    print(" ", bigram_sample())

在分子生成基准测试（如 GuacaMol、MOSES）中，通常用三个核心指标评价一批生成样本的质量：

- **有效性（Validity）**：生成的字符串能被 RDKit 成功解析为合法化学分子的比例；
- **唯一性（Uniqueness）**：合法分子经规范化（Canonicalization）去重后，不重复分子占合法分子的比例（过低说明模型发生了模式崩溃，翻来覆去只写同几个分子）；
- **新颖性（Novelty）**：不重复的合法分子中，未在训练集出现过的新分子比例（过低说明模型只是在死记硬背训练集）。

In [ ]:
canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}

def evaluate(samples, name="model"):
    """计算分子生成模型的三个标准指标：有效性（valid）、唯一性（unique）、新颖性（novel）。"""
    mols = [Chem.MolFromSmiles(s) for s in samples]
    valid = [Chem.MolToSmiles(m) for m in mols if m is not None and m.GetNumAtoms() > 0]
    unique = set(valid)
    novel = unique - canonical_train
    row = {
        "valid": len(valid) / len(samples),
        "unique": len(unique) / max(len(valid), 1),
        "novel": len(novel) / max(len(unique), 1),
    }
    print(f"{name:22s} valid {row['valid']:.3f}  unique {row['unique']:.3f}  novel {row['novel']:.3f}")
    return row

bigram_samples = [bigram_sample() for _ in range(1000)]
bigram_scores = evaluate(bigram_samples, "Bigram baseline")

二元模型的有效率只有 **0.3%** 左右——生成 1000 条字符串，只有两三条碰巧合法。

原因在于 SMILES 语法具有严格的**长程拓扑约束（Long-Range Constraints）**：

1. **圆括号必须成对闭合**：前面打开了几个支链左括号 `(`，后面就必须准确闭合几个右括号 `)`；
2. **成环数字必须成对出现**：在序列开头写下 `c1ccc...` 开启了一个环，跨越十几个甚至几十个词元之后，必须再次写出 `1` 将该环闭合；
3. **芳香性与化合价守恒**：共轭环内的 $\pi$ 电子数与杂原子价态必须在整个环闭合时自洽。

只看前一个词元的二元模型没有任何“历史记忆”，自然无法配对相距几十步的括号和环号。要跨越长距离记住“当前打开了哪几个环、处于哪一层支链括号内”，我们需要引入具备内部记忆状态的循环神经网络（LSTM）。

---
## 4. 构建 SMILES 语言模型（LSTM）

完整的自回归语言模型将整条 SMILES 序列 $S = (s_1, s_2, \ldots, s_T)$ 的联合概率分解为每一步条件概率的连乘积：

$$P(S) = P(s_1) \cdot P(s_2 \mid s_1) \cdot P(s_3 \mid s_1, s_2) \cdots P(s_T \mid s_1, \ldots, s_{T-1})$$

模型架构由三个基础模块串联而成：

| 模块 | 输入与输出形状 | 物理与数学作用 |
|---|---|---|
| `nn.Embedding` | `(batch, L)` $\rightarrow$ `(batch, L, emb_dim)` | 将离散的词元编号映射为连续稠密向量（由网络自动学习每个原子/键符号的语义表征） |
| `nn.LSTM` | `(batch, L, emb_dim)` $\rightarrow$ `(batch, L, hidden_size)` | 沿序列逐步递推更新隐状态 $(h_t, c_t)$，用门控机制筛选并记忆长程未闭合的环号与括号状态 |
| `nn.Linear` | `(batch, L, hidden_size)` $\rightarrow$ `(batch, L, vocab_size)` | 将当前隐状态映射为词表中 23 个词元的未归一化对数几率（logits），经 Softmax 得到下一词元的概率分布 |

In [ ]:
class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, n_layers, batch_first=True, dropout=0.1)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):
        """x: (batch, seq_len) 的词元编号张量；返回每个位置预测下一词元的 logits 及更新后的隐状态。"""
        out, state = self.lstm(self.embedding(x), state)
        return self.linear(out), state

model = SmilesRNN(len(vocab))
print(model)
print("模型总参数量:", sum(p.numel() for p in model.parameters()))

### 4.1 训练机制：Teacher Forcing 与交叉熵损失

训练自回归语言模型时，标准做法是 **Teacher Forcing（教师强制）**：把真实分子序列向左错开一位作为监督目标，让模型在序列的每一个位置 $t$ 上，都基于**真实的前 $t$ 个词元**预测第 $t+1$ 个词元：

```
输入序列 x[:, :-1]:   ^   c   1   c   c   s   c   1
目标序列 x[:, 1:] :   c   1   c   c   s   c   1   $
```

这样一条长度为 50 的 SMILES 字符串就能在一次前向传播中同时提供 50 个分类训练信号。损失函数采用多分类交叉熵 `nn.CrossEntropyLoss(ignore_index=0)`，其中 `ignore_index=0` 确保末尾补齐用的 `<pad>` 位置不产生梯度。

在开始训练前，先检查一个**未经训练的随机初始化模型**在看到起始符 `^` 后的预测概率分布和初始损失：

In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

example = data[:1]
with torch.no_grad():
    logits, _ = model(example[:, :-1])
    probs = torch.softmax(logits[0, 0], dim=-1)

print("未训练模型在起始符 ^ 之后预测的前 5 高概率词元:")
print(pd.Series(probs.numpy(), index=vocab).sort_values(ascending=False).head(5).round(4))
print(f"\n23 个词元均匀分布的理论概率: 1/{len(vocab)} = {1 / len(vocab):.4f}")
init_loss = loss_fn(logits.reshape(-1, len(vocab)), example[:, 1:].reshape(-1)).item()
print(f"未训练模型的初始交叉熵损失: {init_loss:.3f} (均匀瞎猜的理论熵 ln({len(vocab)}) = {np.log(len(vocab)):.3f})")

未训练模型对 23 个词元给出的概率都在 $1/23 \approx 0.0435$ 附近，初始交叉熵损失恰等于 $\ln 23 \approx 3.135$。

---
## 5. 训练模型与加载预训练权重

我们先编写增量自回归采样函数 `sample`，然后在 CEP 数据集上现场训练 3 个 epoch（CPU 上约需 2–3 分钟），每轮结束后采样 200 个分子观察合法率的变化。

注意训练循环中的 `nn.utils.clip_grad_norm_(model.parameters(), 5)`：循环神经网络在长序列反向传播时，梯度连乘容易引发**梯度爆炸（Exploding Gradients）**，对梯度范数进行裁剪是训练 RNN/LSTM 的标准工程保护措施。

In [ ]:
@torch.no_grad()
def sample(model, n=500, temperature=1.0, max_steps=max_len):
    """从语言模型中并行自回归采样 n 条 SMILES 字符串。"""
    model.eval()
    x = torch.full((n, 1), stoi["^"])
    state = None
    out = torch.zeros(n, 0, dtype=torch.long)
    finished = torch.zeros(n, dtype=torch.bool)
    for _ in range(max_steps):
        logits, state = model(x, state)            # 每次仅喂入最新采样的 1 个词元，隐状态 state 携带前文记忆
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        x = torch.multinomial(probs, 1)            # 按条件概率分布随机抽样
        out = torch.cat([out, x], dim=1)
        finished |= (x[:, 0] == stoi["$"])
        if finished.all():
            break
    model.train()

    smiles_list = []
    for row in out.tolist():
        tokens = []
        for t in row:
            if t == stoi["$"]:
                break
            tokens.append(vocab[t])
        smiles_list.append("".join(tokens))
    return smiles_list

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epochs(model, optimizer, rows, n_epochs, lr=1e-3, batch_size=128):
    for group in optimizer.param_groups:
        group["lr"] = lr
    subset = data[rows]
    for epoch in range(n_epochs):
        perm = np.random.permutation(len(subset))
        total = 0.0
        for i in range(0, len(perm), batch_size):
            batch = subset[perm[i:i + batch_size]]
            logits, _ = model(batch[:, :-1])
            loss = loss_fn(logits.reshape(-1, len(vocab)), batch[:, 1:].reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            total += loss.item()
        samples = sample(model, 200)
        valid = sum(Chem.MolFromSmiles(s) is not None for s in samples) / 200
        print(f"Epoch {epoch + 1}: train loss = {total / (len(perm) / batch_size):.3f} | 采样有效率 = {valid:.2f}")

train_epochs(model, optimizer, np.arange(len(data)), n_epochs=3)

仅仅训练 3 轮，交叉熵损失就从 $3.14$ 降到了 $0.40$ 左右，采样有效率从 $0\%$ 跃升至 $40\%\sim 60\%$——模型已经在仅靠预测下一词元的过程中自发习得了圆括号配对与成环数字闭合。

### 5.1 加载训练满 20 轮的预训练模型

为了节省课堂时间并获得充分收敛的生成效果，下面加载预先在相同数据上训练满 20 轮的模型权重 `data/cep_rnn.pt`。

In [ ]:
CKPT = os.path.join(DATA_DIR, "cep_rnn.pt")
rnn = SmilesRNN(len(vocab))
if os.path.exists(CKPT):
    rnn.load_state_dict(torch.load(CKPT, map_location="cpu"))
    print("已加载 20 轮预训练模型权重:", CKPT)
else:
    print("未找到 cep_rnn.pt，继续使用刚才训练 3 轮的模型")
    rnn = model
rnn.eval();

---
## 6. 自回归采样、温度调节与错误诊断

使用加载好的 `rnn` 采样 1000 个分子，计算有效性、唯一性与新颖性，并可视化前 12 个合法分子结构：

In [ ]:
samples = sample(rnn, 1000)
print("前 5 个生成的 SMILES:")
for s in samples[:5]:
    print(" ", s)
rnn_scores = evaluate(samples, "LSTM (T=1.0)")

In [ ]:
valid_mols = [m for m in (Chem.MolFromSmiles(s) for s in samples) if m is not None and m.GetNumAtoms() > 0]
Draw.MolsToGridImage(valid_mols[:12], molsPerRow=4, subImgSize=(260, 200))

对比第 1 节展示的 CEP 训练集分子，模型生成的分子同样是典型的稠环共轭有机半导体骨架，而且超过 $95\%$ 的生成分子合法、彼此不重复、且未在训练集中出现过。

### 6.1 采样温度（Temperature）的权衡

在 `sample` 函数中，我们将每一步输出的 logits 除以**温度参数 $T$** 后再做 Softmax：

$$P(s_t = i \mid s_{<t}) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- **$T < 1$（低温冷却）**：放大高分词元的优势，概率分布变尖锐。模型趋于保守，只走把握最大的主干路径，因此**有效率升高，但多样性（唯一性与新颖性）下降**；
- **$T = 1$（标准温度）**：严格按照模型学到的条件概率分布采样；
- **$T > 1$（高温加热）**：拉平不同词元之间的概率差距，低概率词元也有机会被抽中。模型更具探索性，**唯一性与新颖性维持高位，但更容易拼错括号或违反价态，导致有效率下降**。

In [ ]:
rows = []
for T in [0.5, 0.7, 1.0, 1.2, 1.5]:
    scores = evaluate(sample(rnn, 1000, temperature=T), f"T = {T}")
    rows.append({"T": T, **scores})
temperature_table = pd.DataFrame(rows).set_index("T")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
temperature_table.plot(marker="o", ax=ax)
ax.set_xlabel("Sampling temperature T")
ax.set_ylabel("Fraction")
ax.set_ylim(0, 1.05)
ax.set_title("Effect of sampling temperature on generation metrics")
plt.tight_layout()
plt.show()

---
### 练习 08a-1：诊断无效 SMILES 的两类病因（语法错误 vs. 化学错误）

当 `Chem.MolFromSmiles(s)` 返回 `None` 时，模型究竟错在了哪里？请编写代码从 `sample(rnn, 2000)` 生成的样本中筛选出所有无效 SMILES，并将它们细分为两类：

1. **语法错误（Syntax Error）**：调用 `Chem.MolFromSmiles(s, sanitize=False)`（仅解析字符串拓扑，不做化学合理性检查）就返回 `None`，说明圆括号未配对或成环数字落单；
2. **化学错误（Chemistry Error）**：`sanitize=False` 可以解析出分子图对象 `raw`，但无法通过化学检查（如原子化合价超标、芳香环不满足休克尔规则无法完成单双键交替的凯库勒化 `KekulizeException`）。

统计两类错误各占多少比例，使用 `[p.GetType() for p in Chem.DetectChemistryProblems(raw)]` 统计第二类错误的具体问题类型，并分别打印 3 个具体例子。

In [ ]:
# 练习 08a-1：在这里写你的代码


### 练习 08a-2：贪心解码（Greedy Decoding）vs. 多项式随机采样

为什么在 `sample` 函数中我们要用 `torch.multinomial(probs, 1)` 按概率随机抽签，而不是每一步直接挑概率最大的那个词元（即 $T \to 0$ 时的**贪心解码**）？

请仿照 `sample` 编写一个 `greedy_sample(model, n=10)` 函数，把 `torch.multinomial(probs, 1)` 替换为 `logits[:, -1].argmax(dim=-1, keepdim=True)`，生成 10 个分子并打印出来。观察这 10 个分子是否完全相同，并解释原因。

In [ ]:
# 练习 08a-2：在这里写你的代码


---
## 7. 模型真的学到化学了吗？验证分布复现

回到第 1 节的核心论断：**无条件语言模型学习的是训练数据的统计分布**。我们从宏观物理化学描述符和微观杂环构筑单元两个层面进行验证。

### 7.1 物理化学描述符分布对比

从训练集中随机抽取 2000 个分子，与模型在 $T=1.0$ 下生成的合法分子对比重原子数（`heavy atoms`）、芳香环数（`aromatic rings`）、脂水分配系数（`MolLogP`）和拓扑极性表面积（`TPSA`）的分布直方图：

In [ ]:
generated = sample(rnn, 2000, temperature=1.0)
gen_mols = [m for m in (Chem.MolFromSmiles(s) for s in generated) if m is not None and m.GetNumAtoms() > 0]
train_mols = [mols[i] for i in np.random.RandomState(0).choice(len(mols), 2000, replace=False)]

properties = {
    "heavy atoms": lambda m: m.GetNumHeavyAtoms(),
    "aromatic rings": Descriptors.NumAromaticRings,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, (name, fn) in zip(axes, properties.items()):
    ax.hist([fn(m) for m in train_mols], bins=25, density=True, alpha=0.6, label="CEP training data")
    ax.hist([fn(m) for m in gen_mols], bins=25, density=True, alpha=0.6, label="Generated")
    ax.set_title(name)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

在訓練过程中，我们从未向损失函数输入过任何分子量、环数或极性表面积信息，但生成分子的四个物理化学性质分布与 CEP 训练集几乎完全重合。

### 7.2 共轭构筑单元（Building Blocks）出现频率对比

进一步用子结构匹配（`HasSubstructMatch`）统计有机半导体中 6 种经典共轭杂环单元（噻吩、呋喃、吡咯、硒吩、噻二唑、苯环）在训练集与生成分子中的出现比例：

In [ ]:
fragments = {
    "thiophene": "c1ccsc1",
    "furan": "c1ccoc1",
    "pyrrole": "c1cc[nH]c1",
    "selenophene": "c1cc[se]c1",
    "thiadiazole": "c1nsnc1",
    "benzene": "c1ccccc1",
}
rows = []
for name, smarts in fragments.items():
    patt = Chem.MolFromSmiles(smarts)
    rows.append({
        "fragment": name,
        "CEP training": np.mean([m.HasSubstructMatch(patt) for m in train_mols]),
        "generated": np.mean([m.HasSubstructMatch(patt) for m in gen_mols]),
    })
fragment_table = pd.DataFrame(rows).set_index("fragment")
fragment_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
fragment_table.plot.barh(ax=ax)
ax.set_xlabel("Fraction of molecules containing the fragment")
ax.set_title("Conjugated building blocks: training data vs. generated")
plt.tight_layout()
plt.show()

各个共轭构筑单元在生成集与训练集中的占比高度一致。模型通过学习字符序列的条件转移概率，精准复刻了训练集的化学子结构组成分布。

---
### 练习 08a-3：最近邻 Tanimoto 相似度分布——检验模型是在“近似抄袭”还是在“合理插值”

第 6 节的 `novel` 指标表明 $98\%$ 以上的生成分子不在训练集里，但“规范 SMILES 字符串不完全相同”并不等于“结构不高度雷同”（如果模型只是把训练分子边缘的一个氧原子换成硫原子，`novel` 依然是 100%）。

请使用第 03b 讲学过的 Morgan 分子指纹（半径 2，2048 位）和 `DataStructs.BulkTanimotoSimilarity` 完成以下对比检验：

1. 从训练集中随机抽取 3000 个分子作为参考集 `reference_fps`；
2. 对 500 个合法的生成分子，计算每个分子到参考集的**最大 Tanimoto 相似度**（即最近邻相似度 `gen_sim`）；
3. 建立对照组：从训练集里另外随机抽 500 个分子，计算它们到参考集中除自身外的**最大 Tanimoto 相似度**（`inside_sim`）；
4. 将两条最大相似度分布画在同一张直方图上，并统计相似度 $> 0.9$ 的比例。生成分子与训练集的最近邻距离，和训练分子彼此之间的最近邻距离相比如何？

In [ ]:
# 练习 08a-3：在这里写你的代码


---
## 本节小结

| 核心概念 | 方法与要点 |
|---|---|
| 无条件序列生成本质 | 将分子概率分解为条件概率连乘 $P(S)=\prod_t P(s_t \mid s_{<t})$，拟合训练集的化学结构分布 |
| SMILES 正则分词 | 保持 `[...]`、`Br`、`Cl`、`%dd` 完整性，并用 `assert "".join(tokens) == smiles` 校验可逆性 |
| 特殊控制词元 | `<pad>`（填充补齐）、`^`（起始符 BOS）、`$`（结束符 EOS） |
| 非神经网络基线 | 二元模型（Bigram）有效率仅 ~0.3%，证明圆括号与成环配对等长程拓扑约束必须依赖序列记忆 |
| 语言模型架构与训练 | `nn.Embedding` $\rightarrow$ `nn.LSTM` $\rightarrow$ `nn.Linear`；采用 Teacher Forcing、忽略 `<pad>` 的交叉熵损失与梯度裁剪 |
| 采样与温度调节 | 使用 `state` 增量递推并按多项式分布抽样；低温 $T<1$ 提高有效率但牺牲多样性，高温 $T>1$ 增加多样性但废品率上升 |
| 评估体系 | 有效性（Validity）、唯一性（Uniqueness）、新颖性（Novelty）+ 性质分布与最近邻 Tanimoto 相似度检验 |

在下一节 **`08b_smiles_lstm_aqsoldb.ipynb`** 中，我们将面对更具挑战性的真实研发场景：**当我们手头只有一百多个目标领域的有机半导体分子（远不够从头训练语言模型）时，如何利用通用分子库的预训练语言模型、SMILES 随机重写数据增强、负对数似然（NLL）异常检测以及跨领域迁移微调（Fine-Tuning），实现小样本下的高效分子生成与虚拟筛选。**